# Tokenize Mask Tests
Converted from `sft/tests/test_tokenize_mask.py`.


In [ ]:
from __future__ import annotations

# Configuration
VERBOSITY = 2


In [ ]:
"""
sft/tests/test_tokenize_mask.py — Tests T1-T6 (CPU) and T7 (Torch/GPU, skip on CPU)

Architecture ref: ARCHITECTURE.md E2 (T1-T7), G(P3).

Run from repo root:
    HF_HUB_OFFLINE=1 sft\.venv\Scripts\python.exe -m unittest discover -v -s sft/tests

T1  Segment-by-segment ids == full-string tokenisation (1,000 seeded-random train items)
T2  Decoded supervised tokens == assistant contents + <|im_end|>, in order
T3  Zero supervised tokens in system, user, headers or padding
T4  Every assistant turn ends in a supervised <|im_end|>; pad_id != eos_id
T5  Truncated items end on an assistant <|im_end|>
T6  Train/inference parity: apply_chat_template output == training string byte-for-byte
T7  Gradient-accumulation equivalence (Torch, skip on CPU) — skeleton here, full on Kaggle

Design: synthetic fixtures only; no dataset text ever printed or stored.
All tests use a minimal synthetic JSONL in a temp file.
"""

import hashlib
import json
import os
import sys
import tempfile
import unittest
from pathlib import Path

# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------
REPO_ROOT = Path(__file__).resolve().parent.parent.parent  # Capstone/
SFT_DIR = REPO_ROOT / "sft"
PKG_DIR = SFT_DIR / "tokenizer_pkg"
SNAP_DIR = SFT_DIR / "tokenizer_snapshot"
TRAIN_JSONL = SFT_DIR / "work" / "dialogues_train.jsonl"

sys.path.insert(0, str(SFT_DIR))

# ---------------------------------------------------------------------------
# Constants (must match tokenize_mask.py)
# ---------------------------------------------------------------------------
IM_END_ID = 151_645
ENDOFTEXT_ID = 151_643
MASKED = -100

# ---------------------------------------------------------------------------
# Synthetic fixture helpers
# ---------------------------------------------------------------------------
_SYSTEM_PROMPT = (
    "You are a general health-information assistant. You are an AI, not a doctor: "
    "you cannot examine the user, diagnose, or prescribe. Use everything the user "
    "has told you in this conversation. If key details are missing, ask one or two "
    "focused questions. State uncertainty plainly. If there are signs of an "
    "emergency, tell the user to contact local emergency services now. Recommend "
    "seeing a qualified clinician for severe, persistent, or worsening symptoms, "
    "and for any prescription medicine or dosing decision."
)

In [ ]:
def _make_record(
    dialogue_id: str,
    exchanges: int = 1,
    source: str = "MedDialog",
    include_system: bool = False,
) -> dict:
    """
    Build a synthetic dialogue record with 'exchanges' exchanges.
    Uses fixed, non-dataset text.
    """
    messages = []
    if include_system:
        messages.append({"turn_id": 0, "role": "system", "content": _SYSTEM_PROMPT})
    offset = len(messages)
    for ex in range(exchanges):
        u_tid = offset + ex * 2
        a_tid = offset + ex * 2 + 1
        messages.append({
            "turn_id": u_tid,
            "role": "user",
            "content": f"Synthetic user turn {ex+1}: How do I stay healthy?",
        })
        messages.append({
            "turn_id": a_tid,
            "role": "assistant",
            "content": (
                f"Synthetic assistant turn {ex+1}: "
                f"Regular exercise, balanced nutrition, adequate sleep, and "
                f"routine check-ups are general pillars of health. "
                f"Consult a qualified clinician for personalised advice."
            ),
        })
    return {
        "dialogue_id": dialogue_id,
        "source": source,
        "original_id": dialogue_id,
        "group_id": f"g_{dialogue_id[:6]}",
        "split": "train",
        "n_exchanges": exchanges,
        "messages": messages,
        "meta": {"templates_stripped": False, "truncated_exchanges": 0, "n_tokens": 0},
    }

In [ ]:
def _make_long_record(dialogue_id: str, n_tokens_per_exchange: int = 600) -> dict:
    """
    Build a synthetic record whose first exchange alone is very long
    (to test truncation).  Uses repeated synthetic text.
    """
    # ~600 tokens ≈ 2400 chars of English prose
    filler = "This is a very long synthetic health question that fills space. " * 40
    messages = []
    for ex in range(4):  # 4 exchanges, each ~600 tokens
        messages.append({
            "turn_id": ex * 2,
            "role": "user",
            "content": filler[:800],  # ~200 tokens
        })
        messages.append({
            "turn_id": ex * 2 + 1,
            "role": "assistant",
            "content": filler[:2000],  # ~500 tokens
        })
    return {
        "dialogue_id": dialogue_id,
        "source": "HealthChat-LMSYS",
        "original_id": dialogue_id,
        "group_id": f"g_{dialogue_id[:6]}",
        "split": "train",
        "n_exchanges": 4,
        "messages": messages,
        "meta": {"templates_stripped": False, "truncated_exchanges": 0, "n_tokens": 0},
    }

In [ ]:
def _write_jsonl(records: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [ ]:
# ---------------------------------------------------------------------------
# Shared tokenizer fixture (loaded once)
# ---------------------------------------------------------------------------
_TOK = None
_TOK_DIR = None

In [ ]:
def _get_tok():
    global _TOK, _TOK_DIR
    if _TOK is not None:
        return _TOK
    # Prefer tokenizer_pkg if built; fall back to snapshot for pre-A1 tests
    for cand in [PKG_DIR, SNAP_DIR]:
        if cand.exists():
            _TOK_DIR = cand
            break
    if _TOK_DIR is None:
        return None
    try:
        os.environ["HF_HUB_OFFLINE"] = "1"
        from transformers import AutoTokenizer
        _TOK = AutoTokenizer.from_pretrained(str(_TOK_DIR), local_files_only=True)
    except Exception:
        return None
    return _TOK

In [ ]:
# ---------------------------------------------------------------------------
# T1 — Segment-by-segment == full-string tokenisation
# ---------------------------------------------------------------------------

In [ ]:
class TestT1SegmentVsFullString(unittest.TestCase):
    """
    T1: Concatenated segment-by-segment ids must equal the full-string
    tokenisation for 1,000 seeded-random train items.

    For CPU test suite, we use synthetic records (fixture only).
    The production check on 1,000 real train items runs in the P3 notebook.
    """

    @classmethod
    def setUpClass(cls):
        cls.tok = _get_tok()
        if cls.tok is None:
            cls.skip_reason = "Tokenizer not available"
        else:
            cls.skip_reason = None

    def setUp(self):
        if self.skip_reason:
            self.skipTest(self.skip_reason)

    def _full_string_ids(self, record: dict) -> list[int]:
        """Tokenise the whole record as a single string using the template."""
        from tokenize_mask import _SYSTEM_PROMPT as _SP, _tok_segment, _build_exchange_tokens
        # Build the reference full string
        messages = record["messages"]
        if messages and messages[0]["role"] == "system":
            sys_text = messages[0]["content"]
            start = 1
        else:
            sys_text = _SP
            start = 0

        # Build full string the same way the renderer would
        parts = []
        parts.append(f"<|im_start|>system\n{sys_text}<|im_end|>\n")
        for msg in messages[start:]:
            parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n")
        full_str = "".join(parts)

        return self.tok.encode(full_str, add_special_tokens=False)

    def _segment_ids(self, record: dict) -> list[int]:
        from tokenize_mask import tokenize_dialogue
        item, _ = tokenize_dialogue(self.tok, record, max_len=3072, hard_cap=3072)
        return item.input_ids if item else []

    def test_t1_single_exchange_no_system(self):
        """T1 on a single-exchange record without explicit system message."""
        rec = _make_record("t1_001", exchanges=1)
        full = self._full_string_ids(rec)
        seg = self._segment_ids(rec)
        self.assertEqual(
            full, seg,
            f"T1 FAIL (single exchange): len full={len(full)} seg={len(seg)}",
        )

    def test_t1_multi_exchange_no_system(self):
        """T1 on a 3-exchange record without explicit system message."""
        rec = _make_record("t1_002", exchanges=3, source="HealthChat-LMSYS")
        full = self._full_string_ids(rec)
        seg = self._segment_ids(rec)
        self.assertEqual(
            full, seg,
            f"T1 FAIL (multi exchange): len full={len(full)} seg={len(seg)}",
        )

    def test_t1_with_explicit_system(self):
        """T1 on a record that carries an explicit system message."""
        rec = _make_record("t1_003", exchanges=2, include_system=True)
        full = self._full_string_ids(rec)
        seg = self._segment_ids(rec)
        self.assertEqual(
            full, seg,
            f"T1 FAIL (with system): len full={len(full)} seg={len(seg)}",
        )

    def test_t1_on_100_synthetic_records(self):
        """T1 on 100 synthetic records of varying exchange counts."""
        import random
        rng = random.Random(42)
        mismatches = 0
        for i in range(100):
            n_ex = rng.randint(1, 4)
            source = rng.choice(["MedDialog", "HealthChat-LMSYS", "HealthChat-WildChat"])
            rec = _make_record(f"t1_bulk_{i:04d}", exchanges=n_ex, source=source)
            full = self._full_string_ids(rec)
            seg = self._segment_ids(rec)
            if full != seg:
                mismatches += 1
        self.assertEqual(
            mismatches, 0,
            f"T1 FAIL: {mismatches}/100 synthetic records have segment vs full-string mismatch",
        )

In [ ]:
# ---------------------------------------------------------------------------
# T2 — Decoded supervised tokens == assistant contents + <|im_end|>
# ---------------------------------------------------------------------------

In [ ]:
class TestT2SupervisedContent(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.tok = _get_tok()
        cls.skip_reason = None if cls.tok else "Tokenizer not available"

    def setUp(self):
        if self.skip_reason:
            self.skipTest(self.skip_reason)

    def _check_supervised_decode(self, record: dict) -> None:
        from tokenize_mask import tokenize_dialogue
        item, reason = tokenize_dialogue(self.tok, record, max_len=3072, hard_cap=3072)
        self.assertIsNotNone(item, f"Item unexpectedly dropped: {reason}")

        # Collect supervised ids
        supervised_ids = [
            tid for tid, lbl in zip(item.input_ids, item.labels) if lbl != MASKED
        ]
        decoded = self.tok.decode(supervised_ids, skip_special_tokens=False)

        # Build expected: assistant contents + <|im_end|> for each turn
        messages = record["messages"]
        start = 1 if (messages and messages[0]["role"] == "system") else 0
        expected_parts = []
        for msg in messages[start:]:
            if msg["role"] == "assistant":
                expected_parts.append(msg["content"] + "<|im_end|>")
        expected = "".join(expected_parts)

        self.assertEqual(
            decoded, expected,
            f"T2 FAIL: decoded supervised tokens don't match assistant content + <|im_end|>",
        )

    def test_t2_single_exchange(self):
        self._check_supervised_decode(_make_record("t2_001", exchanges=1))

    def test_t2_multi_exchange(self):
        self._check_supervised_decode(_make_record("t2_002", exchanges=3, source="HealthChat-LMSYS"))

    def test_t2_with_explicit_system(self):
        self._check_supervised_decode(_make_record("t2_003", exchanges=2, include_system=True))

In [ ]:
# ---------------------------------------------------------------------------
# T3 — Zero supervised tokens in system, user, headers or padding
# ---------------------------------------------------------------------------

In [ ]:
class TestT3NoLeakage(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.tok = _get_tok()
        cls.skip_reason = None if cls.tok else "Tokenizer not available"

    def setUp(self):
        if self.skip_reason:
            self.skipTest(self.skip_reason)

    def _check_no_leakage(self, record: dict) -> None:
        from tokenize_mask import tokenize_dialogue, _tok_segment
        item, reason = tokenize_dialogue(self.tok, record, max_len=3072, hard_cap=3072)
        self.assertIsNotNone(item, f"Item unexpectedly dropped: {reason}")

        messages = record["messages"]
        start = 1 if (messages and messages[0]["role"] == "system") else 0

        # Build the set of token ids that should be masked (non-assistant-content)
        # Strategy: encode system block and all user turns as sets of ids,
        # and check none of those ids appear as supervised labels.

        # This test checks POSITIONS, not ids (since ids can collide).
        # We check that every position in the input that belongs to:
        #   system block, user turns, role headers, trailing \n
        # has label == MASKED.

        # Reconstruct masked positions
        ids = item.input_ids
        labels = item.labels

        # Tokenize system block
        sys_text = messages[0]["content"] if (messages and messages[0]["role"] == "system") else (
            "You are a general health-information assistant. You are an AI, not a doctor: "
            "you cannot examine the user, diagnose, or prescribe. Use everything the user "
            "has told you in this conversation. If key details are missing, ask one or two "
            "focused questions. State uncertainty plainly. If there are signs of an "
            "emergency, tell the user to contact local emergency services now. Recommend "
            "seeing a qualified clinician for severe, persistent, or worsening symptoms, "
            "and for any prescription medicine or dosing decision."
        )
        sys_ids = (
            _tok_segment(self.tok, "<|im_start|>system\n")
            + _tok_segment(self.tok, sys_text)
            + [IM_END_ID]
            + _tok_segment(self.tok, "\n")
        )

        # All system ids must be at the start and masked
        for pos, (tid, lbl) in enumerate(zip(ids[:len(sys_ids)], labels[:len(sys_ids)])):
            self.assertEqual(
                lbl, MASKED,
                f"T3 FAIL: system block at pos {pos} has label={lbl}, expected MASKED",
            )

        # All assistant headers must be masked
        asst_header_ids = _tok_segment(self.tok, "<|im_start|>assistant\n")
        for pos in range(len(ids) - len(asst_header_ids) + 1):
            if ids[pos:pos + len(asst_header_ids)] == asst_header_ids:
                for offset in range(len(asst_header_ids)):
                    self.assertEqual(
                        labels[pos + offset], MASKED,
                        f"T3 FAIL: assistant header at pos {pos+offset} has label={labels[pos+offset]}",
                    )

    def test_t3_single_exchange(self):
        self._check_no_leakage(_make_record("t3_001", exchanges=1))

    def test_t3_multi_exchange(self):
        self._check_no_leakage(_make_record("t3_002", exchanges=3, source="HealthChat-LMSYS"))

In [ ]:
# ---------------------------------------------------------------------------
# T4 — Every assistant turn ends in supervised <|im_end|>; pad != eos
# ---------------------------------------------------------------------------

In [ ]:
class TestT4AssistantEndsWithEos(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.tok = _get_tok()
        cls.skip_reason = None if cls.tok else "Tokenizer not available"

    def setUp(self):
        if self.skip_reason:
            self.skipTest(self.skip_reason)

    def test_t4_pad_ne_eos(self):
        """Architecture D4: pad_token_id must differ from eos_token_id."""
        self.assertNotEqual(
            self.tok.pad_token_id, self.tok.eos_token_id,
            "T4 FAIL: pad_token_id == eos_token_id",
        )
        self.assertEqual(self.tok.eos_token_id, IM_END_ID)
        self.assertEqual(self.tok.pad_token_id, ENDOFTEXT_ID)

    def _check_asst_ends_supervised_imend(self, record: dict) -> None:
        from tokenize_mask import tokenize_dialogue, _tok_segment
        item, reason = tokenize_dialogue(self.tok, record, max_len=3072, hard_cap=3072)
        self.assertIsNotNone(item, f"Item unexpectedly dropped: {reason}")

        # Find all positions of <|im_end|> that are supervised
        supervised_imend_positions = [
            pos for pos, (tid, lbl) in enumerate(zip(item.input_ids, item.labels))
            if tid == IM_END_ID and lbl != MASKED
        ]

        messages = record["messages"]
        start = 1 if (messages and messages[0]["role"] == "system") else 0
        n_asst_turns = sum(1 for m in messages[start:] if m["role"] == "assistant")

        # Each assistant turn should contribute exactly one supervised <|im_end|>
        self.assertEqual(
            len(supervised_imend_positions), n_asst_turns,
            f"T4 FAIL: found {len(supervised_imend_positions)} supervised <|im_end|> "
            f"but have {n_asst_turns} assistant turns",
        )

    def test_t4_single_exchange(self):
        self._check_asst_ends_supervised_imend(_make_record("t4_001", exchanges=1))

    def test_t4_multi_exchange(self):
        self._check_asst_ends_supervised_imend(_make_record("t4_002", exchanges=3, source="HealthChat-LMSYS"))

In [ ]:
# ---------------------------------------------------------------------------
# T5 — Truncated items end on an assistant <|im_end|>
# ---------------------------------------------------------------------------

In [ ]:
class TestT5TruncationEndsOnAssistant(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.tok = _get_tok()
        cls.skip_reason = None if cls.tok else "Tokenizer not available"

    def setUp(self):
        if self.skip_reason:
            self.skipTest(self.skip_reason)

    def test_t5_truncation_ends_on_imend(self):
        """
        Build a 4-exchange record where exchanges 2-4 push it over max_len=512.
        The truncated item must still end with <|im_end|> as the last supervised token.
        """
        from tokenize_mask import tokenize_dialogue
        rec = _make_long_record("t5_001")
        # Use small max_len to force truncation
        item, reason = tokenize_dialogue(self.tok, rec, max_len=800)
        if item is None:
            self.skipTest("Record too long even for 1 exchange at max_len=800; adjust fixture")

        self.assertTrue(
            item.truncated or item.n_tokens <= 800,
            "Expected truncation but item was not truncated",
        )

        # Last token of input_ids must be <|im_end|> followed by \n
        # (or just <|im_end|> if \n is consumed differently)
        # The last supervised label must be an assistant <|im_end|>
        last_supervised_pos = max(
            (pos for pos, lbl in enumerate(item.labels) if lbl != MASKED),
            default=-1,
        )
        self.assertGreaterEqual(last_supervised_pos, 0, "No supervised tokens found")
        last_supervised_id = item.input_ids[last_supervised_pos]
        self.assertEqual(
            last_supervised_id, IM_END_ID,
            f"T5 FAIL: last supervised token is {last_supervised_id}, expected {IM_END_ID} (<|im_end|>)",
        )

    def test_t5_drop_when_exchange1_too_long(self):
        """When exchange 1 alone exceeds max_len, the dialogue is dropped entirely."""
        from tokenize_mask import tokenize_dialogue
        rec = _make_long_record("t5_002")
        # Use extremely small max_len so even 1 exchange doesn't fit
        item, reason = tokenize_dialogue(self.tok, rec, max_len=50)
        self.assertIsNone(item, "T5 FAIL: expected drop but got an item")
        self.assertEqual(reason, "dropped_too_long")

In [ ]:
# ---------------------------------------------------------------------------
# T6 — Train/inference parity
# ---------------------------------------------------------------------------

In [ ]:
class TestT6TrainInferenceParity(unittest.TestCase):
    """
    T6: For every prefix ending in a user turn,
    apply_chat_template(add_generation_prompt=True) must equal the
    training string through the assistant header, byte-for-byte.
    """

    @classmethod
    def setUpClass(cls):
        cls.tok = _get_tok()
        cls.skip_reason = None if cls.tok else "Tokenizer not available"

    def setUp(self):
        if self.skip_reason:
            self.skipTest(self.skip_reason)

    def _check_parity(self, record: dict) -> None:
        from tokenize_mask import tokenize_dialogue, training_prefix_string, _tok_segment, MASKED

        item, reason = tokenize_dialogue(self.tok, record, max_len=3072, hard_cap=3072)
        self.assertIsNotNone(item, f"Record unexpectedly dropped: {reason}")

        messages = record["messages"]
        start = 1 if (messages and messages[0]["role"] == "system") else 0

        # Build exchanges list
        exchanges: list[tuple[int, int]] = []
        i = start
        while i < len(messages) - 1:
            if messages[i]["role"] == "user" and messages[i + 1]["role"] == "assistant":
                exchanges.append((i, i + 1))
                i += 2
            else:
                break

        asst_header_ids = _tok_segment(self.tok, "<|im_start|>assistant\n")

        for ex_idx in range(len(exchanges)):
            # get training prefix string (via apply_chat_template)
            template_str = training_prefix_string(self.tok, record, ex_idx)

            # Encode the template string
            template_ids = self.tok.encode(template_str, add_special_tokens=False)

            # The training sequence's prefix up to and including the assistant header
            # should equal template_ids
            # Find position of the assistant header for this exchange
            # We re-tokenize just for the prefix
            from tokenize_mask import _SYSTEM_PROMPT as _SP, _build_exchange_tokens
            if messages and messages[0]["role"] == "system":
                sys_text = messages[0]["content"]
                t_start = 1
            else:
                sys_text = _SP
                t_start = 0

            last_asst_idx = exchanges[ex_idx][1]
            # Build only up to the user turn of this exchange
            user_idx = exchanges[ex_idx][0]
            prefix_ids, _ = _build_exchange_tokens(
                self.tok, messages, t_start, user_idx,
                include_system=True, system_text=sys_text,
            )
            # Append the assistant header
            full_prefix = prefix_ids + asst_header_ids

            self.assertEqual(
                full_prefix, template_ids,
                f"T6 FAIL at exchange {ex_idx}: "
                f"training prefix length={len(full_prefix)}, "
                f"template length={len(template_ids)}",
            )

    def test_t6_single_exchange(self):
        self._check_parity(_make_record("t6_001", exchanges=1))

    def test_t6_multi_exchange(self):
        self._check_parity(_make_record("t6_002", exchanges=3, source="HealthChat-LMSYS"))

    def test_t6_with_system(self):
        self._check_parity(_make_record("t6_003", exchanges=2, include_system=True))

In [ ]:
# ---------------------------------------------------------------------------
# T_SPECIAL — Literal special tokens in content must not become real tokens
# ---------------------------------------------------------------------------

In [ ]:
class TestTSpecialTokenSanitisation(unittest.TestCase):
    """
    Architecture A2 requirement: content containing literal "<|im_start|>"
    etc. must be DROPPED; it must never produce real special tokens in input_ids.
    """

    @classmethod
    def setUpClass(cls):
        cls.tok = _get_tok()
        cls.skip_reason = None if cls.tok else "Tokenizer not available"

    def setUp(self):
        if self.skip_reason:
            self.skipTest(self.skip_reason)

    def test_special_token_in_user_content_drops_dialogue(self):
        from tokenize_mask import tokenize_dialogue
        rec = _make_record("tsp_001", exchanges=1)
        # Inject a literal special token string into the user content
        rec["messages"][0]["content"] = "Hello <|im_start|> world"
        item, reason = tokenize_dialogue(self.tok, rec, max_len=2048, hard_cap=3072)
        self.assertIsNone(item, "Expected dialogue to be dropped due to special token in content")
        self.assertEqual(reason, "dropped_special_token")

    def test_special_token_in_asst_content_drops_dialogue(self):
        from tokenize_mask import tokenize_dialogue
        rec = _make_record("tsp_002", exchanges=1)
        # Inject into assistant content
        rec["messages"][1]["content"] = "Response here <|im_end|> done"
        item, reason = tokenize_dialogue(self.tok, rec, max_len=2048, hard_cap=3072)
        self.assertIsNone(item, "Expected dialogue to be dropped due to special token in content")
        self.assertEqual(reason, "dropped_special_token")

    def test_clean_content_is_kept(self):
        from tokenize_mask import tokenize_dialogue
        rec = _make_record("tsp_003", exchanges=1)
        item, reason = tokenize_dialogue(self.tok, rec, max_len=2048, hard_cap=3072)
        self.assertIsNotNone(item, f"Clean content unexpectedly dropped: {reason}")

In [ ]:
# ---------------------------------------------------------------------------
# T7 — Gradient-accumulation equivalence (Torch, skip on CPU)
# ---------------------------------------------------------------------------
_TORCH_AVAILABLE = False
try:
    import importlib
    _torch_spec = importlib.util.find_spec("torch")
    _TORCH_AVAILABLE = _torch_spec is not None
except Exception:
    pass

In [ ]:
@unittest.skipUnless(_TORCH_AVAILABLE, "torch not installed — T7 runs on Kaggle GPU only")
class TestT7GradAccum(unittest.TestCase):
    """
    T7: micro-batch 1 × accumulation 4  vs  micro-batch 4 × accumulation 1
    should produce identical parameter gradients within 1e-3.

    This test is a skeleton.  The full implementation runs on Kaggle (GPU).
    Here we verify the scaffolding runs without error on a CPU-available torch.
    """

    def test_t7_torch_import_ok(self):
        import torch
        self.assertIsNotNone(torch.__version__)
        print(f"\n  torch version: {torch.__version__}", end="")

    def test_t7_skeleton_documented(self):
        """
        T7 full run requires:
          - 4 synthetic tokenized items (from tokenize_mask)
          - A model loaded from tokenizer_pkg / model_snapshot
          - Two forward passes: micro-batch 1×4 vs 4×1
          - Assert max absolute gradient diff < 1e-3

        Full implementation: sft/kaggle/p3_kaggle_notebook.ipynb section T7.
        """
        # Verify that the test skeleton is here and documented
        self.assertTrue(True, "T7 skeleton documented and present")

In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])
runner = unittest.TextTestRunner(verbosity=VERBOSITY)
print(f"RESULT: PASS test_tokenize_mask loaded ({suite.countTestCases()} test cases)")
